# Flight Price Competition — EDA, Leakage-Safe CV, and Non-NN Ensembles

This notebook is designed for **Kaggle Notebooks** or local Python. It uses **training labels only** and predicts the untouched test set. Select submission by the competition's official metric (RMSE assumed for now). Precomputed full 34-chart EDA and benchmark files are included in the companion package.

In [ ]:
from pathlib import Path
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings('ignore')
# Auto-detect input location on Kaggle or local runtime.
candidates = [Path('/mnt/data')] + [p.parent for p in Path('/kaggle/input').rglob('train.csv')] if Path('/kaggle/input').exists() else [Path('/mnt/data')]
data_dir = next((d for d in candidates if all((d/n).exists() for n in ['train.csv','test.csv','sample_submission.csv'])), None)
if data_dir is None: raise FileNotFoundError('Place train.csv, test.csv, and sample_submission.csv in a single dataset directory.')
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
train = pd.read_csv(data_dir/'train.csv')
test  = pd.read_csv(data_dir/'test.csv')
sub = pd.read_csv(data_dir/'sample_submission.csv')
assert sub.id.equals(test.id)
print('Train:', train.shape, 'Test:',test.shape, 'Destination:', output_dir)
display(train.head())

## Step 1. Univariate audit

Study distribution shape, rare categories, missingness, and plausible outliers. Class-aware summaries prevent high Business prices from being mistaken for arbitrary outliers.

In [ ]:
display(pd.DataFrame({'train_missing_pct':train.isna().mean()*100,'test_missing_pct':test.isna().mean().reindex(train.columns)*100}).round(2))
display(train.describe(include='all').T)
fig, ax = plt.subplots(1,3,figsize=(17,4))
sns.histplot(train.price,bins=75,ax=ax[0]);ax[0].set_title('Price (skewed)')
sns.histplot(np.log1p(train.price),bins=65,ax=ax[1]);ax[1].set_title('log1p(price)')
sns.countplot(data=train,x='flight_class',ax=ax[2]);ax[2].set_title('Class counts')
plt.tight_layout();plt.show()
for col in ['airline','source_city','destination_city','departure_time','arrival_time','stops']:
 print('\n',col, '\n',train[col].value_counts(dropna=False).head(20).to_string())

## Step 2. Bivariate pricing relationships

Compare ticket classes, route origins, lead time, stopping pattern and duration. The supplied full EDA report also includes joint timing effects and population drift.

In [ ]:
display(train.groupby('flight_class').price.agg(['size','mean','median','std','min','max']))
fig,axes=plt.subplots(2,2,figsize=(15,10))
sns.boxenplot(data=train,x='flight_class',y='price',ax=axes[0,0]);axes[0,0].set_title('Price by class')
sns.barplot(data=train,x='stops',y='price',hue='flight_class',estimator=np.median,errorbar=None,ax=axes[0,1]);axes[0,1].set_title('Stops × class: median price')
byday=train.groupby(['days_left','flight_class']).price.median().reset_index()
sns.lineplot(data=byday,x='days_left',y='price',hue='flight_class',ax=axes[1,0]);axes[1,0].set_title('Booking lead time')
sns.scatterplot(data=train.sample(min(3500,len(train)),random_state=42),x='duration',y='price',hue='flight_class',alpha=.4,s=11,ax=axes[1,1]);axes[1,1].set_title('Duration × class')
plt.tight_layout();plt.show()

## Step 3. Multivariate EDA, covariance and missingness

Route×class price matrices reveal strong conditional structure. Reconstruct an airline-level proxy from the flight prefix; never leak labels by computing target means outside CV.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(15,5))
for i,clazz in enumerate(['Economy','Business']):
 tab=train[train.flight_class==clazz].pivot_table(index='source_city',columns='destination_city',values='price',aggfunc='median')
 sns.heatmap(tab,annot=True,fmt='.0f',cmap='YlGnBu',ax=ax[i]);ax[i].set_title(clazz+' median price by route')
plt.tight_layout();plt.show()
train['flight_prefix']=train.flight.str.split().str[0]
print('Missing airline rows recoverable from known prefix:',train.loc[train.airline.isna(),'flight_prefix'].isin(train.loc[train.airline.notna(),'flight_prefix']).mean())
print('Test flight codes not in train:',(~test.flight.isin(train.flight)).sum())
print('Duplicate training rows ignoring ID:',train.drop(columns=['id','flight_prefix']).duplicated().sum())

## Step 4. Feature engineering (unsupervised and deterministic)

ID and target never enter the model. All categorical encoders, imputers, and class-based target mappings are fit within each training fold. Missingness is explicitly encoded.

In [ ]:
CATS=['flight','flight_prefix','source_city','destination_city','departure_time','arrival_time','stops','flight_class','route','route_class','time_pair','prefix_class','stops_route']
NUM=['duration','days_left','flight_number','duration_missing','airline_missing','days_left_sq','days_left_inv','duration_per_stop','stops_num']
def make_features(x):
 x=x.copy()
 x['flight_prefix']=x.flight.str.extract(r'^([^ ]+)')[0].fillna('UNK')
 x['flight_number']=pd.to_numeric(x.flight.str.extract(r'(\d+)\s*$')[0],errors='coerce')
 x['duration_missing']=x.duration.isna().astype(int)
 x['airline_missing']=x.airline.isna().astype(int)
 x['route']=x.source_city.astype(str)+'__'+x.destination_city.astype(str)
 x['route_class']=x.route+'__'+x.flight_class.astype(str)
 x['time_pair']=x.departure_time.astype(str)+'__'+x.arrival_time.astype(str)
 x['prefix_class']=x.flight_prefix+'__'+x.flight_class.astype(str)
 x['stops_route']=x.stops.astype(str)+'__'+x.route
 x['stops_num']=x.stops.map({'zero':0,'one':1,'two_or_more':2}).astype(int)
 x['days_left_sq']=x.days_left.astype(float)**2
 x['days_left_inv']=1/(x.days_left.astype(float)+1)
 x['duration_per_stop']=x.duration/(x.stops_num+1)
 for c in CATS:x[c]=x[c].fillna('__UNKNOWN__').astype(str)
 return x[CATS+NUM]
X=make_features(train);X_test=make_features(test);y=train.price.to_numpy(float)
display(X.head())

## Step 5. Train-only five-fold model comparison and test prediction

Compares bootstrap-based Random Forest, Extra Trees, boosted LightGBM and CatBoost. For each fold, train fitting never sees validation rows or test targets. Reduce `MODEL_NAMES` for faster smoke tests. No neural networks.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor,RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

def strata(y,classes):
 d=pd.DataFrame({'target':y,'clazz':classes})
 series=[]
 for cl in d.clazz.unique():
  ind=d.index[d.clazz==cl]
  q=pd.qcut(d.loc[ind,'target'],5,labels=False,duplicates='drop')
  series.append(pd.Series(cl+'_'+q.astype(str).to_numpy(),index=ind))
 return pd.concat(series).sort_index().to_numpy()
def make_model(name):
 if name in ['extra_trees_tuned','random_forest']:
  enc=ColumnTransformer([('categorical',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),CATS),('numeric',SimpleImputer(strategy='median'),NUM)])
  model=(ExtraTreesRegressor(n_estimators=260,min_samples_leaf=3,max_features=.85,n_jobs=4,random_state=42)
         if name=='extra_trees_tuned' else RandomForestRegressor(n_estimators=180,max_depth=24,min_samples_leaf=3,max_features=.9,n_jobs=4,random_state=42))
  return Pipeline([('prep',enc),('model',model)])
 if name=='lightgbm':return LGBMRegressor(n_estimators=850,learning_rate=.045,num_leaves=23,min_child_samples=35,subsample=.85,subsample_freq=1,colsample_bytree=.88,reg_lambda=7,reg_alpha=.1,verbosity=-1,n_jobs=4,random_state=42)
 if name=='catboost':return CatBoostRegressor(iterations=1150,depth=7,learning_rate=.045,l2_leaf_reg=6,loss_function='RMSE',random_seed=42,thread_count=4,verbose=False,allow_writing_files=False)
 raise ValueError(name)
def predict_fold(name,model,ta,va,tt,y_train):
 if name=='lightgbm':
  ta,va,tt=ta.copy(),va.copy(),tt.copy()
  for c in CATS:
   dtype=pd.CategoricalDtype(categories=pd.Index(ta[c].unique()))
   for d in [ta,va,tt]:d[c]=d[c].astype(dtype)
  model.fit(ta,y_train,categorical_feature=CATS)
 elif name=='catboost':model.fit(ta,y_train,cat_features=CATS)
 else:model.fit(ta,y_train)
 return np.clip(model.predict(va),1,None),np.clip(model.predict(tt),1,None)

MODEL_NAMES=['extra_trees_tuned','random_forest','lightgbm','catboost']
folds=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=2026).split(X,strata(y,train.flight_class)))
oof={};testpred={};score_rows=[]
for name in MODEL_NAMES:
 p=np.zeros(len(y));q=[]
 for fold,(a,b) in enumerate(folds,1):
  model=make_model(name)
  pred,test_fold_pred=predict_fold(name,model,X.iloc[a],X.iloc[b],X_test,y[a]);p[b]=pred;q.append(test_fold_pred)
  print(name,'fold',fold,'RMSE',round(np.sqrt(mean_squared_error(y[b],pred)),2))
 oof[name]=p;testpred[name]=np.mean(q,axis=0)
 score_rows.append({'model':name,'rmse':np.sqrt(mean_squared_error(y,p)),'mae':mean_absolute_error(y,p),'r2':r2_score(y,p)})
display(pd.DataFrame(score_rows).sort_values('rmse'))

## Step 6. Regularized positive-weight, class-specific ensemble

Fit the **second-level blending weights on out-of-fold training predictions**, not the test set. Test predictions are averaged from the five independently trained fold models. Because meta-weights are tuned on these OOF labels, the displayed full OOF ensemble metric is slightly optimistic—use a nested validation for final competitive rigor.

In [ ]:
from scipy.optimize import minimize
model_names=MODEL_NAMES
A=np.column_stack([oof[n] for n in model_names]);B=np.column_stack([testpred[n] for n in model_names])
prior=np.array([.55,.10,.20,.15]);scale=30000.0

def blend_weights(A,y,prior):
 def objective(w):return np.mean(((A@w-y)/scale)**2)+.002*np.sum((w-prior)**2)
 result=minimize(objective,prior,method='SLSQP',bounds=[(0,1)]*len(prior),constraints=[{'type':'eq','fun':lambda w:w.sum()-1}],options={'maxiter':300,'ftol':1e-12})
 if not result.success:print('Optimizer warning:',result.message)
 w=np.maximum(result.x,0)
 return w/w.sum()
blended_oof=np.zeros(len(y));blended_test=np.zeros(len(test))
for clazz in ['Economy','Business']:
 a=(train.flight_class==clazz).to_numpy();b=(test.flight_class==clazz).to_numpy()
 weights=blend_weights(A[a],y[a],prior)
 print(clazz,dict(zip(model_names,np.round(weights,3))))
 blended_oof[a]=A[a]@weights;blended_test[b]=B[b]@weights
print('OOF blend RMSE:',np.sqrt(mean_squared_error(y,blended_oof)))
print('OOF blend MAE:',mean_absolute_error(y,blended_oof))
submission=pd.DataFrame({'id':test.id,'price':blended_test})
assert submission.columns.tolist()==['id','price'] and submission.id.equals(sub.id)
assert submission.price.notna().all() and np.isfinite(submission.price).all() and (submission.price>0).all()
filename=output_dir/'submission_class_specific_ensemble.csv'
submission.to_csv(filename,index=False)
print('Saved:',filename);display(submission.head())

### Additional tests to consider

- Confirm the **official competition metric**: RMSE, MAE and RMSLE rank models differently.
- Compare feature-group ablations, grouped-flight holdouts, bootstrap resamples, and additive blending under the **same folds**.
- Avoid selecting repeatedly based on the public leaderboard; preserve an untouched validation protocol for generalization claims.
- **Classification algorithms** (Logistic Regression / Naive Bayes classifier) are not suitable for the original continuous target. Neural networks are excluded.
